In [3]:
# === BIBLIOTHÈQUES ===
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE

# === MODÈLES ===
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# === DATA ===
df = pd.read_csv(r"C:\Users\zizou\OneDrive\Desktop\stage 3ème\day 2\csvfiles\dataframefinale.csv", sep=';', encoding='utf-8-sig')
df = df.dropna()

# === CIBLE ===
X = df.drop(columns=["Montant"])
y = df["Montant"]

# Encodage des variables catégorielles si nécessaire
X = pd.get_dummies(X, drop_first=True)

# === SPLIT TRAIN/TEST ===
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [4]:
df

,dataloadingdate,Jour,Mois,NumeroSemaine,Trimestre,JourSemaineNum,JourSemaine,ISIN,Libellé,Nombre de Titres,Montant,Echéance,Taux
0,01/07/2025,1,7,27,3,1,Tuesday,TN0008000739,"BTA 7,4% Fevrier 2030",1459.0,1.500,62.0,7.60
1,01/07/2025,1,7,27,3,1,Tuesday,TN0008000739,"BTA 7,4% Fevrier 2030",291.0,0.300,183.0,7.50
2,01/07/2025,1,7,27,3,1,Tuesday,TNMCPXLL1EE2,EMP NAT 2023 T4 CB TV,50000.0,5.000,13.0,8.60
3,01/07/2025,1,7,27,3,1,Tuesday,TNMCPXLL1EE2,EMP NAT 2023 T4 CB TV,10000.0,1.000,7.0,8.60
4,01/07/2025,1,7,27,3,1,Tuesday,TN0008000606,"BTA 6,7% Avril 2028",5660.0,5.742,31.0,9.10
...,...,...,...,...,...,...,...,...,...,...,...,...,...
19990,31/12/2024,31,12,1,4,1,Tuesday,TN0008000812,"BTA 7,5% 13/12/2028",215.0,0.200,50.0,9.00
19991,31/12/2024,31,12,1,4,1,Tuesday,TNX0K9990B08,EMP NAT 2024 T2 CB TF,5243.0,0.556,15.0,7.49
19992,31/12/2024,31,12,1,4,1,Tuesday,TNX0K9990B08,EMP NAT 2024 T2 CB TF,28288.0,3.000,31.0,8.99
19993,31/12/2024,31,12,1,4,1,Tuesday,TNX0K9990B08,EMP NAT 2024 T2 CB TF,48701.0,5.165,31.0,8.99


In [5]:
import math

def evaluate_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    rmse = math.sqrt(mean_squared_error(y_test, preds))
    return rmse


In [6]:
models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "RandomForest": RandomForestRegressor(random_state=42),
    "GradientBoosting": GradientBoostingRegressor(random_state=42)
}


In [7]:
print("=== Étape 1: RMSE sans transformation ===")
for name, model in models.items():
    rmse = evaluate_model(model, X_train, X_test, y_train, y_test)
    print(f"{name}: RMSE = {rmse:.4f}")


=== Étape 1: RMSE sans transformation ===
LinearRegression: RMSE = 7221710.6647
Ridge: RMSE = 6.0179
Lasso: RMSE = 7.0584
RandomForest: RMSE = 2.9043
GradientBoosting: RMSE = 4.3996


In [8]:
print("=== Étape 2: RMSE avec normalisation ===")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

for name, model in models.items():
    rmse = evaluate_model(model, X_train_scaled, X_test_scaled, y_train, y_test)
    print(f"{name}: RMSE = {rmse:.4f}")


=== Étape 2: RMSE avec normalisation ===
LinearRegression: RMSE = 5500995893956.9121
Ridge: RMSE = 6.0332
Lasso: RMSE = 7.2029
RandomForest: RMSE = 2.9058
GradientBoosting: RMSE = 4.3996


In [9]:
print("=== Étape 3: RMSE avec RFE (sélection de features) ===")
from sklearn.base import clone

for name, model in models.items():
    try:
        model_clone = clone(model)
        selector = RFE(estimator=model_clone, n_features_to_select=10, step=1)
        selector.fit(X_train_scaled, y_train)
        X_train_rfe = selector.transform(X_train_scaled)
        X_test_rfe = selector.transform(X_test_scaled)
        
        rmse = evaluate_model(model_clone, X_train_rfe, X_test_rfe, y_train, y_test)
        print(f"{name}: RMSE = {rmse:.4f}")
    except Exception as e:
        print(f"{name}: Erreur lors de la sélection de features : {e}")


=== Étape 3: RMSE avec RFE (sélection de features) ===
LinearRegression: RMSE = 2202956943.3576
Ridge: RMSE = 6.3364
Lasso: RMSE = 7.2029


KeyboardInterrupt: 